## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-22-22-48-36 +0000,nyt,ICE Officers Arrest 2 Passengers Boarding a Fl...,https://www.nytimes.com/2026/11/22/us/ice-arre...
1,2026-11-22-22-42-27 +0000,nypost,Trump faces pressure as deadline over gray wol...,https://nypost.com/2026/11/22/us-news/trump-fa...
2,2026-11-22-22-40-11 +0000,nypost,Humiliation for California college after hitti...,https://nypost.com/2026/11/22/us-news/uc-santa...
3,2026-11-22-22-37-43 +0000,bbc,'Danger seems greater all the time': Ukrainian...,https://www.bbc.co.uk/news/videos/cre3dvd1k9y1...
4,2026-11-22-22-34-44 +0000,latimes,Missing 19-year-old Marine found dead after we...,https://www.latimes.com/california/story/2026-...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2517/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
8,trump,21
138,house,11
19,california,11
162,new,10
375,deal,7


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
107,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...,53
37,2026-11-22-20-48-02 +0000,chitri,Chuck Hubbard defeats Rep. Virginia Foxx after...,https://www.chicagotribune.com/2026/11/22/chuc...,45
138,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...,45
80,2026-11-22-17-58-17 +0000,nypost,Former NYC mayor Bill de Blasio warns Dems not...,https://nypost.com/2026/11/22/us-news/de-blasi...,44
26,2026-11-22-21-12-07 +0000,nypost,Texas GOP senators openly duke it out over Tru...,https://nypost.com/2026/11/22/us-news/texas-se...,44


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
107,53,2026-11-22-15-27-09 +0000,nypost,"Iran taunts Trump, shows off underground ‘miss...",https://nypost.com/2026/11/22/us-news/iran-sho...
37,45,2026-11-22-20-48-02 +0000,chitri,Chuck Hubbard defeats Rep. Virginia Foxx after...,https://www.chicagotribune.com/2026/11/22/chuc...
4,38,2026-11-22-22-34-44 +0000,latimes,Missing 19-year-old Marine found dead after we...,https://www.latimes.com/california/story/2026-...
92,37,2026-11-22-17-11-00 +0000,cbc,COP31 climate talks collapse after member stat...,https://www.cbc.ca/news/world/cop31-talks-coll...
67,29,2026-11-22-18-42-39 +0000,wapo,House Democrats will start next Congress with ...,https://www.washingtonpost.com/politics/2026/1...
150,28,2026-11-22-10-00-00 +0000,wsj,Richer and more adventurous than their predece...,https://www.wsj.com/personal-finance/retiremen...
24,25,2026-11-22-21-15-35 +0000,nypost,NYC pol sets record straight as viral ‘eyes’ v...,https://nypost.com/2026/11/22/us-news/nyc-pol-...
17,25,2026-11-22-21-39-42 +0000,wapo,Sen. Kennedy urges scrutiny of Lutnick over Ch...,https://www.washingtonpost.com/politics/2026/1...
169,25,2026-11-22-06-46-36 +0000,nypost,Endurance athlete Lewis Pugh braves Strait of ...,https://nypost.com/2026/11/22/world-news/briti...
18,24,2026-11-22-21-38-38 +0000,nypost,Dem House leader Hakeem Jeffries refuses to ru...,https://nypost.com/2026/11/22/us-news/hakeem-j...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
